# Training: Real or Fake

**Target mapping:** `fake = 0`, `real = 1`. `ImageFolder` assigns class indices alphabetically, so the class folders must be named `fake` and `real` (or `FAKE` and `REAL`) consistently in every split. This notebook validates the mapping before training.

**Models:** CNN, ResNet50, VGG16, EfficientNet-B0, and ViT-B/16. Each model has its own training, evaluation, confusion-matrix, ROC-curve, and saving cells.

Before running this notebook, run `03_data_split_train_val_test_split.ipynb` and confirm that `data/split_real_fake` contains `train/fake`, `train/real`, `val/fake`, `val/real`, `test/fake`, and `test/real`.

Run the notebook from top to bottom. If you do not want to train a model, skip all cells for that model, including its build, training, evaluation, confusion matrix, ROC curve, and saving steps.

Pretrained weights require an internet connection on the first run. Settings: CNN = 15 epochs; transfer-learning models = 10 epochs with early stopping; batch size = 32; AdamW optimizer; and `ReduceLROnPlateau`. The pretrained models fine-tune selected final layers to reduce GPU memory use.

#### Import the required libraries:

In [2]:
import copy
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve, classification_report)
import plotly.express as px
import plotly.graph_objects as go

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti


### Load the dataset

`ImageFolder` treats each class subfolder as a target label and assigns indices in alphabetical order. With `fake` and `real` folders, the expected mapping is `fake = 0` and `real = 1`.

In [3]:
# Set the project directory.
PROJECT_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final")

# Set the data and model directories.
DATA_DIR = PROJECT_DIR / "data"
TASK = "real_fake"

SPLIT_DIR = DATA_DIR / "split_real_fake"

# Save trained models in the existing project-level models directory.
MODEL_DIR = PROJECT_DIR / "models"/ "real_fake"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CHARTS_DIR = PROJECT_DIR / "charts" / "real_fake"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Define the training, validation, and test directories.
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"

# Check that all split directories exist.
split_dirs = [TRAIN_DIR, VAL_DIR, TEST_DIR]
missing_splits = [
    str(folder) for folder in split_dirs
    if not folder.is_dir()
]

if missing_splits:
    raise FileNotFoundError(
        "The following split directories are missing: "
        + ", ".join(missing_splits)
        + ". Run 03_data_split_train_val_test_split.ipynb first."
    )

# Check that each split contains the required REAL and FAKE class folders.
for split_dir in split_dirs:
    class_names = {
        folder.name.lower()
        for folder in split_dir.iterdir()
        if folder.is_dir()
    }

    if not {"fake", "real"}.issubset(class_names):
        raise FileNotFoundError(
            f"Both 'fake' and 'real' folders must exist in: {split_dir}"
        )

# Set training parameters.
BATCH_SIZE = 32  # Reduce to 16 if a CUDA out-of-memory error occurs.
PRETRAINED = True  # Set to False to train without pretrained ImageNet weights.
SEED = 42

# Set random seeds for reproducibility.
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Data directory:", DATA_DIR)
print("Split directory:", SPLIT_DIR)
print("Model save directory:", MODEL_DIR)
print("Charts save directory:", CHARTS_DIR)
print("Training, validation, and test folders are ready.")

Data directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data
Split directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\split_real_fake
Model save directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\real_fake
Charts save directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\real_fake
Training, validation, and test folders are ready.


# Data Preprocessing:

### Transform Compose:

Red channel: mean = 0.485, std = 0.229

Green channel: mean = 0.456, std = 0.224

Blue channel: mean = 0.406, std = 0.225

### Train:

In [4]:
train_transform = transforms.Compose([
                                        transforms.Resize((224, 224)),
                                        transforms.RandomHorizontalFlip(p=0.5),
                                        transforms.RandomRotation(degrees=10),
                                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                             std=[0.229, 0.224, 0.225])
                                    ])

### Test/Val:

In [5]:
val_test_transform = transforms.Compose([
                                transforms.Resize((224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225])
                                        ])

In [6]:
# Create datasets for training, training-set evaluation, validation, and testing.
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_eval_dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_test_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

# Confirm that every split uses the same target-to-index mapping.
expected_class_to_idx = {"fake": 0, "real": 1}
def normalize_class_mapping(class_to_idx):
    return {name.lower(): index for name, index in class_to_idx.items()}

train_mapping = normalize_class_mapping(train_dataset.class_to_idx)
val_mapping = normalize_class_mapping(val_dataset.class_to_idx)
test_mapping = normalize_class_mapping(test_dataset.class_to_idx)

for split_name, mapping in [
    ("training", train_mapping),
    ("validation", val_mapping),
    ("test", test_mapping)
]:
    if mapping != expected_class_to_idx:
        raise ValueError(
            f"Incorrect target mapping in the {split_name} split: {mapping}. "
            "Expected {'fake': 0, 'real': 1}. Check the class folder names."
        )

class_names = train_dataset.classes
num_classes = len(class_names)

if num_classes != 2:
    raise ValueError(f"Expected exactly two classes (fake and real), but found: {class_names}")

print("Training class mapping:", train_dataset.class_to_idx)
print("Validation class mapping:", val_dataset.class_to_idx)
print("Test class mapping:", test_dataset.class_to_idx)
print("Target mapping verified: fake = 0, real = 1")

image, label = train_dataset[0]
print("Example target index:", label, "->", class_names[label])
print("Image type:", type(image))
print("Image shape:", image.shape)


Training class mapping: {'fake': 0, 'real': 1}
Validation class mapping: {'fake': 0, 'real': 1}
Test class mapping: {'fake': 0, 'real': 1}
Target mapping verified: fake = 0, real = 1
Example target index: 0 -> fake
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])


# Data Loader:

In [7]:
# Use num_workers=0 to avoid common multiprocessing issues on Windows.
train_loader      = DataLoader(train_dataset,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader        = DataLoader(val_dataset,        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader       = DataLoader(test_dataset,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 112
Val batches: 16
Test batches: 32


In [8]:
images, labels = next(iter(train_loader))
print("Batch of images shape:", images.shape)
print("Batch of labels shape:", labels.shape)

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])


### Class weights

Class weights help reduce the effect of class imbalance by assigning a larger loss weight to classes with fewer training images.

In [9]:
# Count training images in each class (fake = 0, real = 1).
counts = np.bincount(train_dataset.targets, minlength=num_classes)

# Fail clearly if any expected class has no training images.
if np.any(counts == 0):
    missing_classes = [
        class_names[i] for i, count in enumerate(counts) if count == 0
    ]
    raise ValueError(f"Classes with no training images: {missing_classes}")

# Inverse-frequency weights: classes with fewer images receive larger weights.
weights = len(train_dataset) / (num_classes * counts)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

for name, count, weight in zip(class_names, counts, weights):
    print(f"{name:10s} images = {count:5d}   weight = {weight:.3f}")


fake       images =  1749   weight = 1.018
real       images =  1812   weight = 0.983


# Model Building

Models are created on the CPU and moved to the GPU only for training to reduce GPU memory usage.

### Model -1 - CNN Model:

In [10]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.block4 = block(128, 256)
        self.block5 = block(256, 512)

        self.dropout_conv = nn.Dropout(0.3)
        self.gap = nn.AdaptiveAvgPool2d(1)       # Reduce each feature map to a 1x1 value.

        self.fc1 = nn.Linear(512, 256)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.dropout_conv(x)

        x = self.gap(x)
        x = x.view(x.size(0), -1)                # Flatten features to [batch_size, 512].

        x = torch.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)                          # Return logits; CrossEntropyLoss applies log-softmax internally.
        return x

cnn_model = CNN(num_classes)
print(cnn_model)

CNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), 

### Model - 2 - Transfer Learning (ResNet 50):

In [11]:
# Use the official ResNet50 ImageNet weights when PRETRAINED=True.
resnet_weights = models.ResNet50_Weights.DEFAULT if PRETRAINED else None
resnet_model = models.resnet50(weights=resnet_weights)

# Freeze all pretrained layers.
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace the final classifier with a trainable binary classifier.
resnet_model.fc = nn.Sequential(
    nn.Linear(resnet_model.fc.in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# Fine-tune the final ResNet block.
for param in resnet_model.layer4.parameters():
    param.requires_grad = True

print(resnet_model.fc)


Sequential(
  (0): Linear(in_features=2048, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=2, bias=True)
)


### Model - 3 - Transfer Learning (VGG16):

In [12]:
# Use the official VGG16 ImageNet weights when PRETRAINED=True.
vgg_weights = models.VGG16_Weights.DEFAULT if PRETRAINED else None
vgg_model = models.vgg16(weights=vgg_weights)

# Freeze all pretrained layers.
for param in vgg_model.parameters():
    param.requires_grad = False

# Replace the classifier. VGG16's feature output is 512 * 7 * 7 = 25088.
vgg_model.classifier = nn.Sequential(
    nn.Linear(512 * 7 * 7, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# Fine-tune the final VGG convolution block.
for param in vgg_model.features[24:].parameters():
    param.requires_grad = True

print(vgg_model.classifier)


Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=2, bias=True)
)


### Model - 4 - Transfer Learning (EfficientNet-B0):

In [13]:
# Use the official EfficientNet-B0 ImageNet weights when PRETRAINED=True.
effnet_weights = models.EfficientNet_B0_Weights.DEFAULT if PRETRAINED else None
effnet_model = models.efficientnet_b0(weights=effnet_weights)

# Freeze all pretrained layers.
for param in effnet_model.parameters():
    param.requires_grad = False

# Replace the classifier with a trainable binary classifier.
effnet_in_features = effnet_model.classifier[1].in_features
effnet_model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(effnet_in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# Fine-tune the final EfficientNet feature block.
for param in effnet_model.features[7:].parameters():
    param.requires_grad = True

print(effnet_model.classifier)


Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=256, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=2, bias=True)
)


### Model - 5 - Vision Transformer (ViT):

In [14]:
# Use the official ViT-B/16 ImageNet weights when PRETRAINED=True.
vit_weights = models.ViT_B_16_Weights.DEFAULT if PRETRAINED else None
vit_model = models.vit_b_16(weights=vit_weights)

# Freeze all pretrained layers.
for param in vit_model.parameters():
    param.requires_grad = False

# Replace the classification head.
vit_model.heads = nn.Sequential(
    nn.Linear(vit_model.heads[0].in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# Fine-tune the final transformer encoder block and final normalization.
for param in vit_model.encoder.layers[-1].parameters():
    param.requires_grad = True
for param in vit_model.encoder.ln.parameters():
    param.requires_grad = True

print(vit_model.heads)


Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=2, bias=True)
)


# Model Training:

### Training function

- Each epoch performs training followed by validation.
- The learning rate is reduced when validation loss stops improving.
- Training stops early if validation loss does not improve for five consecutive epochs, and the best weights are restored.

In [15]:
def train_model(model, optimizer, criterion, num_epochs):

    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss": []
    }

    best_val_loss = float("inf")
    best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    patience = 5
    epochs_without_improvement = 0

    # Reduce learning rate if validation loss stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    for epoch in range(num_epochs):

        # Training
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / total
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)

        # Validation
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss)

        scheduler.step(val_loss)

        print(
            f"Epoch {epoch + 1}/{num_epochs}: "
            f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
            f"Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}"
        )

        # Keep the best weights and apply early stopping.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    model.load_state_dict(best_weights)
    return history

### CNN Model:

In [16]:
# Train cnn_model.
cnn_model = cnn_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)

cnn_history = train_model(
    cnn_model, optimizer, criterion, num_epochs=15
)

# Move to CPU after training to free GPU memory.
cnn_model = cnn_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/15: 100%|██████████| 112/112 [01:27<00:00,  1.27it/s]


Epoch 1/15: Train Loss=0.1192, Val Loss=0.0522, Train Acc=0.9607, Val Acc=0.9823


Epoch 2/15: 100%|██████████| 112/112 [01:17<00:00,  1.45it/s]


Epoch 2/15: Train Loss=0.0645, Val Loss=0.0622, Train Acc=0.9803, Val Acc=0.9764


Epoch 3/15: 100%|██████████| 112/112 [01:09<00:00,  1.61it/s]


Epoch 3/15: Train Loss=0.0430, Val Loss=0.0074, Train Acc=0.9890, Val Acc=1.0000


... (20 outputs left out)

Epoch 14/15: 100%|██████████| 112/112 [01:11<00:00,  1.56it/s]


Epoch 14/15: Train Loss=0.0133, Val Loss=0.0002, Train Acc=0.9961, Val Acc=1.0000


Epoch 15/15: 100%|██████████| 112/112 [01:13<00:00,  1.52it/s]


Epoch 15/15: Train Loss=0.0037, Val Loss=0.0005, Train Acc=0.9992, Val Acc=1.0000


In [17]:
# Save the trained Custom CNN model.
model_path = MODEL_DIR / "custom_cnn_model_real_fake.pth"
torch.save(cnn_model.state_dict(), model_path)
print(f"Custom CNN model saved to: {model_path}")


Custom CNN model saved to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\real_fake\custom_cnn_model_real_fake.pth


### ResNet 50:

In [18]:
# Train resnet_model.
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW([{"params": resnet_model.layer4.parameters(), "lr": 1e-5}, 
                         {"params": resnet_model.fc.parameters(), "lr": 1e-3}], weight_decay=0.01)

resnet_history = train_model(
    resnet_model, optimizer, criterion, num_epochs=10
)

# Move to CPU after training to free GPU memory.
resnet_model = resnet_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/10: 100%|██████████| 112/112 [01:24<00:00,  1.33it/s]


Epoch 1/10: Train Loss=0.0258, Val Loss=0.0000, Train Acc=0.9944, Val Acc=1.0000


Epoch 2/10: 100%|██████████| 112/112 [01:25<00:00,  1.31it/s]


Epoch 2/10: Train Loss=0.0006, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


Epoch 3/10: 100%|██████████| 112/112 [01:25<00:00,  1.30it/s]


Epoch 3/10: Train Loss=0.0002, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


... (10 outputs left out)

Epoch 9/10: 100%|██████████| 112/112 [01:24<00:00,  1.33it/s]


Epoch 9/10: Train Loss=0.0000, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


Epoch 10/10: 100%|██████████| 112/112 [01:26<00:00,  1.30it/s]


Epoch 10/10: Train Loss=0.0000, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


In [19]:
# Save the trained ResNet50 model.
model_path = MODEL_DIR / "resnet50_model_real_fake.pth"
torch.save(resnet_model.state_dict(), model_path)
print(f"ResNet50 model saved to: {model_path}")


ResNet50 model saved to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\real_fake\resnet50_model_real_fake.pth


### VGG16:

In [20]:
# Train vgg_model.
vgg_model = vgg_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW([{"params": vgg_model.features[24:].parameters(), "lr": 1e-5}, 
                         {"params": vgg_model.classifier.parameters(), "lr": 1e-3}], weight_decay=0.01)

vgg_history = train_model(
    vgg_model, optimizer, criterion, num_epochs=10
)

# Move to CPU after training to free GPU memory.
vgg_model = vgg_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/10: 100%|██████████| 112/112 [01:35<00:00,  1.17it/s]


Epoch 1/10: Train Loss=0.0069, Val Loss=0.0000, Train Acc=0.9944, Val Acc=1.0000


Epoch 2/10: 100%|██████████| 112/112 [01:32<00:00,  1.21it/s]


Epoch 2/10: Train Loss=0.0052, Val Loss=0.0000, Train Acc=0.9992, Val Acc=1.0000


Epoch 3/10: 100%|██████████| 112/112 [01:32<00:00,  1.21it/s]


Epoch 3/10: Train Loss=0.0069, Val Loss=0.0000, Train Acc=0.9994, Val Acc=1.0000


... (2 outputs left out)

Epoch 5/10: 100%|██████████| 112/112 [01:31<00:00,  1.22it/s]


Epoch 5/10: Train Loss=0.0000, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


Epoch 6/10: 100%|██████████| 112/112 [01:31<00:00,  1.22it/s]


Epoch 6/10: Train Loss=0.0000, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000
Early stopping at epoch 6


In [21]:
# Save the trained VGG16 model.
model_path = MODEL_DIR / "vgg16_model_real_fake.pth"
torch.save(vgg_model.state_dict(), model_path)
print(f"VGG16 model saved to: {model_path}")


VGG16 model saved to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\real_fake\vgg16_model_real_fake.pth


### EfficientNet-B0:

In [22]:
# Train effnet_model.
effnet_model = effnet_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW([{"params": effnet_model.features[7:].parameters(), "lr": 1e-5}, 
                         {"params": effnet_model.classifier.parameters(), "lr": 1e-3}], weight_decay=0.01)

effnet_history = train_model(
    effnet_model, optimizer, criterion, num_epochs=10
)

# Move to CPU after training to free GPU memory.
effnet_model = effnet_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/10: 100%|██████████| 112/112 [01:09<00:00,  1.60it/s]


Epoch 1/10: Train Loss=0.0282, Val Loss=0.0025, Train Acc=0.9930, Val Acc=0.9980


Epoch 2/10: 100%|██████████| 112/112 [01:08<00:00,  1.63it/s]


Epoch 2/10: Train Loss=0.0029, Val Loss=0.0000, Train Acc=0.9983, Val Acc=1.0000


Epoch 3/10: 100%|██████████| 112/112 [01:07<00:00,  1.66it/s]


Epoch 3/10: Train Loss=0.0071, Val Loss=0.0000, Train Acc=0.9978, Val Acc=1.0000


... (10 outputs left out)

Epoch 9/10: 100%|██████████| 112/112 [01:08<00:00,  1.63it/s]


Epoch 9/10: Train Loss=0.0004, Val Loss=0.0000, Train Acc=0.9997, Val Acc=1.0000


Epoch 10/10: 100%|██████████| 112/112 [01:07<00:00,  1.67it/s]


Epoch 10/10: Train Loss=0.0016, Val Loss=0.0001, Train Acc=0.9997, Val Acc=1.0000


In [23]:
# Save the trained EfficientNet-B0 model.
model_path = MODEL_DIR / "efficientnet_b0_model_real_fake.pth"
torch.save(effnet_model.state_dict(), model_path)
print(f"EfficientNet-B0 model saved to: {model_path}")


EfficientNet-B0 model saved to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\real_fake\efficientnet_b0_model_real_fake.pth


### ViT:

In [24]:
# Train vit_model.
vit_model = vit_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW([{"params": vit_model.encoder.layers[-1].parameters(), "lr": 1e-5}, 
                         {"params": vit_model.encoder.ln.parameters(), "lr": 1e-5}, 
                         {"params": vit_model.heads.parameters(), "lr": 1e-3}], weight_decay=0.05)

vit_history = train_model(
    vit_model, optimizer, criterion, num_epochs=10
)

# Move to CPU after training to free GPU memory.
vit_model = vit_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 1/10: 100%|██████████| 112/112 [02:43<00:00,  1.46s/it]


Epoch 1/10: Train Loss=0.0172, Val Loss=0.0004, Train Acc=0.9955, Val Acc=1.0000


Epoch 2/10: 100%|██████████| 112/112 [02:43<00:00,  1.46s/it]


Epoch 2/10: Train Loss=0.0002, Val Loss=0.0002, Train Acc=1.0000, Val Acc=1.0000


Epoch 3/10: 100%|██████████| 112/112 [02:43<00:00,  1.46s/it]


Epoch 3/10: Train Loss=0.0001, Val Loss=0.0001, Train Acc=1.0000, Val Acc=1.0000


... (10 outputs left out)

Epoch 9/10: 100%|██████████| 112/112 [02:48<00:00,  1.50s/it]


Epoch 9/10: Train Loss=0.0000, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


Epoch 10/10: 100%|██████████| 112/112 [02:49<00:00,  1.51s/it]


Epoch 10/10: Train Loss=0.0000, Val Loss=0.0000, Train Acc=1.0000, Val Acc=1.0000


In [25]:
# Save the trained ViT-B/16 model.
model_path = MODEL_DIR / "vit_model_real_fake.pth"
torch.save(vit_model.state_dict(), model_path)
print(f"ViT-B/16 model saved to: {model_path}")


ViT-B/16 model saved to: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\real_fake\vit_model_real_fake.pth


### Training curves (overfitting check):

In [26]:
histories = {}
for name, var in [("CNN", "cnn"), ("ResNet50", "resnet"), ("VGG16", "vgg"), ("EfficientNet_B0", "effnet"), ("ViT_B16", "vit")]:
    if var + "_history" in globals():
        histories[name] = globals()[var + "_history"]

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["train_loss"], mode="lines", name=name + " train")
    fig.add_scatter(y=h["val_loss"], mode="lines", line=dict(dash="dash"), name=name + " val")
fig.update_layout(height=450, width=850, title="Loss per Epoch (solid = train, dashed = validation)",
                  xaxis_title="Epoch", yaxis_title="Loss")
fig.show()

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["val_acc"], mode="lines", name=name)
fig.update_layout(height=450, width=850, title="Validation Accuracy per Epoch",
                  xaxis_title="Epoch", yaxis_title="Accuracy")
fig.show()

# Model Evaluation:

In [27]:
def evaluate_model(model, loader, dataset_name="Set", show_report=False):
    model = model.to(device)
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(probs.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    model = model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    all_labels = np.asarray(all_labels)
    all_preds = np.asarray(all_preds)
    all_probs = np.asarray(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    # ROC-AUC requires both classes to be present in this split.
    if np.unique(all_labels).size == 2:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    else:
        auc = float("nan")
        print(f"Warning: AUC is undefined for {dataset_name}; only one class is present.")

    print(f"--- {dataset_name} Metrics ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}\n")

    if show_report:
        print(classification_report(
            all_labels, all_preds, labels=[0, 1],
            target_names=class_names, zero_division=0
        ))

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs
    }


### CNN Model Result:

In [28]:
print("===== CNN RESULTS =====\n")
cnn_train_results = evaluate_model(cnn_model, train_eval_loader, "Train")
cnn_val_results = evaluate_model(cnn_model, val_loader, "Validation")
cnn_test_results = evaluate_model(cnn_model, test_loader, "Test", show_report=True)

===== CNN RESULTS =====

--- Train Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Validation Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

              precision    recall  f1-score   support

        fake       1.00      1.00      1.00       497
        real       1.00      1.00      1.00       521

    accuracy                           1.00      1018
   macro avg       1.00      1.00      1.00      1018
weighted avg       1.00      1.00      1.00      1018



### ResNet 50 Model Result:

In [29]:
print("===== RESNET50 RESULTS =====\n")
resnet_train_results = evaluate_model(resnet_model, train_eval_loader, "Train")
resnet_val_results = evaluate_model(resnet_model, val_loader, "Validation")
resnet_test_results = evaluate_model(resnet_model, test_loader, "Test", show_report=True)

===== RESNET50 RESULTS =====

--- Train Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Validation Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

              precision    recall  f1-score   support

        fake       1.00      1.00      1.00       497
        real       1.00      1.00      1.00       521

    accuracy                           1.00      1018
   macro avg       1.00      1.00      1.00      1018
weighted avg       1.00      1.00      1.00      1018



### VGG16 Model Result:

In [30]:
print("===== VGG16 RESULTS =====\n")
vgg_train_results = evaluate_model(vgg_model, train_eval_loader, "Train")
vgg_val_results = evaluate_model(vgg_model, val_loader, "Validation")
vgg_test_results = evaluate_model(vgg_model, test_loader, "Test", show_report=True)

===== VGG16 RESULTS =====

--- Train Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Validation Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

              precision    recall  f1-score   support

        fake       1.00      1.00      1.00       497
        real       1.00      1.00      1.00       521

    accuracy                           1.00      1018
   macro avg       1.00      1.00      1.00      1018
weighted avg       1.00      1.00      1.00      1018



### EfficientNet-B0 Model Result:

In [31]:
print("===== EFFICIENTNET_B0 RESULTS =====\n")
effnet_train_results = evaluate_model(effnet_model, train_eval_loader, "Train")
effnet_val_results = evaluate_model(effnet_model, val_loader, "Validation")
effnet_test_results = evaluate_model(effnet_model, test_loader, "Test", show_report=True)

===== EFFICIENTNET_B0 RESULTS =====

--- Train Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Validation Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

              precision    recall  f1-score   support

        fake       1.00      1.00      1.00       497
        real       1.00      1.00      1.00       521

    accuracy                           1.00      1018
   macro avg       1.00      1.00      1.00      1018
weighted avg       1.00      1.00      1.00      1018



### ViT Model Result:

In [32]:
print("===== VIT_B16 RESULTS =====\n")
vit_train_results = evaluate_model(vit_model, train_eval_loader, "Train")
vit_val_results = evaluate_model(vit_model, val_loader, "Validation")
vit_test_results = evaluate_model(vit_model, test_loader, "Test", show_report=True)

===== VIT_B16 RESULTS =====

--- Train Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Validation Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
AUC-ROC:   1.0000

              precision    recall  f1-score   support

        fake       1.00      1.00      1.00       497
        real       1.00      1.00      1.00       521

    accuracy                           1.00      1018
   macro avg       1.00      1.00      1.00      1018
weighted avg       1.00      1.00      1.00      1018



### Result Summary:

In [33]:
def create_results_summary(train_results, val_results, test_results):

    return pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"],
        "Train": [train_results["accuracy"], train_results["precision"], train_results["recall"],
                  train_results["f1"], train_results["auc"]],
        "Validation": [val_results["accuracy"], val_results["precision"], val_results["recall"],
                       val_results["f1"], val_results["auc"]],
        "Test": [test_results["accuracy"], test_results["precision"], test_results["recall"],
                 test_results["f1"], test_results["auc"]]
    })

In [34]:
model_list = [("CNN", "cnn"), ("ResNet50", "resnet"), ("VGG16", "vgg"), ("EfficientNet_B0", "effnet"), ("ViT_B16", "vit")]

for name, var in model_list:
    if var + "_test_results" in globals():
        print(f"\n===== {name.upper()} RESULTS =====\n")
        print(create_results_summary(globals()[var + "_train_results"],
                                     globals()[var + "_val_results"],
                                     globals()[var + "_test_results"]))


===== CNN RESULTS =====

      Metric  Train  Validation  Test
0   Accuracy    1.0         1.0   1.0
1  Precision    1.0         1.0   1.0
2     Recall    1.0         1.0   1.0
3   F1-Score    1.0         1.0   1.0
4    AUC-ROC    1.0         1.0   1.0

===== RESNET50 RESULTS =====

      Metric  Train  Validation  Test
0   Accuracy    1.0         1.0   1.0
1  Precision    1.0         1.0   1.0
2     Recall    1.0         1.0   1.0
3   F1-Score    1.0         1.0   1.0
4    AUC-ROC    1.0         1.0   1.0

===== VGG16 RESULTS =====

      Metric  Train  Validation  Test
0   Accuracy    1.0         1.0   1.0
1  Precision    1.0         1.0   1.0
2     Recall    1.0         1.0   1.0
3   F1-Score    1.0         1.0   1.0
4    AUC-ROC    1.0         1.0   1.0

===== EFFICIENTNET_B0 RESULTS =====

      Metric  Train  Validation  Test
0   Accuracy    1.0         1.0   1.0
1  Precision    1.0         1.0   1.0
2     Recall    1.0         1.0   1.0
3   F1-Score    1.0         1.0   1.0
4  

# Confusion Matrix:

The class order is `fake = 0`, `real = 1`, so the axis labels are `['FAKE', 'REAL']`.

### CNN Model:

In [35]:
# Train Confusion Matrix
cm_cnn_train = confusion_matrix(
                        cnn_train_results["labels"],
                        cnn_train_results["preds"]
                    )

cnn_train_fig = px.imshow(
                        cm_cnn_train,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Train'
                    )

cnn_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_train_fig.show()
joblib.dump(cnn_train_fig, CHARTS_DIR / "confusion_matrix_real_fake_cnn_train.pkl")

# Validation Confusion Matrix
cm_cnn_val = confusion_matrix(
                        cnn_val_results["labels"],
                        cnn_val_results["preds"]
                    )

cnn_val_fig = px.imshow(
                        cm_cnn_val,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Validation'
                    )

cnn_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_val_fig.show()
joblib.dump(cnn_val_fig, CHARTS_DIR / "confusion_matrix_real_fake_cnn_val.pkl")

# Test Confusion Matrix
cm_cnn_test = confusion_matrix(
                        cnn_test_results["labels"],
                        cnn_test_results["preds"]
                    )

cnn_test_fig = px.imshow(
                        cm_cnn_test,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Test'
                    )

cnn_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_test_fig.show()
joblib.dump(cnn_test_fig, CHARTS_DIR / "confusion_matrix_real_fake_cnn_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\confusion_matrix_real_fake_cnn_test.pkl']

### ResNet50 Model:

In [36]:
# Train Confusion Matrix
cm_resnet_train = confusion_matrix(
                        resnet_train_results["labels"],
                        resnet_train_results["preds"]
                    )

resnet_train_fig = px.imshow(
                        cm_resnet_train,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Train'
                    )

resnet_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_train_fig.show()
joblib.dump(resnet_train_fig, CHARTS_DIR / "confusion_matrix_real_fake_ResNet50_train.pkl")

# Validation Confusion Matrix
cm_resnet_val = confusion_matrix(
                        resnet_val_results["labels"],
                        resnet_val_results["preds"]
                    )

resnet_val_fig = px.imshow(
                        cm_resnet_val,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Validation'
                    )

resnet_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_val_fig.show()
joblib.dump(resnet_val_fig, CHARTS_DIR / "confusion_matrix_real_fake_ResNet50_val.pkl")

# Test Confusion Matrix
cm_resnet_test = confusion_matrix(
                        resnet_test_results["labels"],
                        resnet_test_results["preds"]
                    )

resnet_test_fig = px.imshow(
                        cm_resnet_test,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Test'
                    )

resnet_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_test_fig.show()
joblib.dump(resnet_test_fig, CHARTS_DIR / "confusion_matrix_real_fake_ResNet50_test.pkl")



['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\confusion_matrix_real_fake_ResNet50_test.pkl']

### VGG16 Model:

In [37]:
# Train Confusion Matrix
cm_vgg_train = confusion_matrix(
                        vgg_train_results["labels"],
                        vgg_train_results["preds"]
                    )

vgg_train_fig = px.imshow(
                        cm_vgg_train,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Train'
                    )

vgg_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_train_fig.show()
joblib.dump(vgg_train_fig, CHARTS_DIR / "confusion_matrix_real_fake_VGG_16_train.pkl")

# Validation Confusion Matrix
cm_vgg_val = confusion_matrix(
                        vgg_val_results["labels"],
                        vgg_val_results["preds"]
                    )

vgg_val_fig = px.imshow(
                        cm_vgg_val,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Validation'
                    )

vgg_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_val_fig.show()
joblib.dump(vgg_val_fig, CHARTS_DIR / "confusion_matrix_real_fake_VGG_16_val.pkl")

# Test Confusion Matrix
cm_vgg_test = confusion_matrix(
                        vgg_test_results["labels"],
                        vgg_test_results["preds"]
                    )

vgg_test_fig = px.imshow(
                        cm_vgg_test,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Test'
                    )

vgg_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_test_fig.show()
joblib.dump(vgg_test_fig, CHARTS_DIR / "confusion_matrix_real_fake_VGG_16_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\confusion_matrix_real_fake_VGG_16_test.pkl']

### EfficientNet_B0 Model:

In [38]:
# Train Confusion Matrix
cm_effnet_train = confusion_matrix(
                        effnet_train_results["labels"],
                        effnet_train_results["preds"]
                    )

effnet_train_fig = px.imshow(
                        cm_effnet_train,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Train'
                    )

effnet_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_train_fig.show()
joblib.dump(effnet_train_fig, CHARTS_DIR / "confusion_matrix_real_fake_EffNet50_train.pkl")

# Validation Confusion Matrix
cm_effnet_val = confusion_matrix(
                        effnet_val_results["labels"],
                        effnet_val_results["preds"]
                    )

effnet_val_fig = px.imshow(
                        cm_effnet_val,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Validation'
                    )

effnet_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_val_fig.show()
joblib.dump(effnet_val_fig, CHARTS_DIR / "confusion_matrix_real_fake_EffNet50_val.pkl")

# Test Confusion Matrix
cm_effnet_test = confusion_matrix(
                        effnet_test_results["labels"],
                        effnet_test_results["preds"]
                    )

effnet_test_fig = px.imshow(
                        cm_effnet_test,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Test'
                    )

effnet_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_test_fig.show()
joblib.dump(effnet_test_fig, CHARTS_DIR / "confusion_matrix_real_fake_EffNet50_test.pkl")


['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\confusion_matrix_real_fake_EffNet50_test.pkl']

### ViT_B16 Model:

In [39]:
# Train Confusion Matrix
cm_vit_train = confusion_matrix(
                        vit_train_results["labels"],
                        vit_train_results["preds"]
                    )

vit_train_fig = px.imshow(
                        cm_vit_train,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Train'
                    )

vit_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_train_fig.show()
joblib.dump(vit_train_fig, CHARTS_DIR / "confusion_matrix_real_fake_ViT_B16_train.pkl")

# Validation Confusion Matrix
cm_vit_val = confusion_matrix(
                        vit_val_results["labels"],
                        vit_val_results["preds"]
                    )

vit_val_fig = px.imshow(
                        cm_vit_val,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Validation'
                    )

vit_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_val_fig.show()
joblib.dump(vit_val_fig, CHARTS_DIR / "confusion_matrix_real_fake_ViT_B16_val.pkl")

# Test Confusion Matrix
cm_vit_test = confusion_matrix(
                        vit_test_results["labels"],
                        vit_test_results["preds"]
                    )

vit_test_fig = px.imshow(
                        cm_vit_test,
                        text_auto=True,
                        x=['FAKE', 'REAL'],      # fake = 0, real = 1
                        y=['FAKE', 'REAL'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Test'
                    )

vit_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_test_fig.show()
joblib.dump(vit_test_fig, CHARTS_DIR / "confusion_matrix_real_fake_ViT_B16_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\confusion_matrix_real_fake_ViT_B16_test.pkl']

# ROC Curve

Train, validation, and test ROC curves with the AUC score of each. The curves use the predicted probability of class 1 (`real`).

### CNN Model:

In [40]:
fpr_tr, tpr_tr, _ = roc_curve(
                    cnn_train_results["labels"],
                    cnn_train_results["probs"][:, 1]      # probability of class 1 (real)
                )

fpr_va, tpr_va, _ = roc_curve(
                    cnn_val_results["labels"],
                    cnn_val_results["probs"][:, 1]
                )

fpr_te, tpr_te, _ = roc_curve(
                    cnn_test_results["labels"],
                    cnn_test_results["probs"][:, 1]
                )

fig = go.Figure()

fig.add_scatter(
                x=fpr_tr,
                y=tpr_tr,
                mode='lines',
                name=f'Train (AUC={cnn_train_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_va,
                y=tpr_va,
                mode='lines',
                name=f'Validation (AUC={cnn_val_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={cnn_test_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
                )

fig.update_layout(
                  height=500, width=800,
                  title='ROC Curve - Custom CNN',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
                  )

fig.show()
joblib.dump(fig, CHARTS_DIR / "real_fake_ROC_Curve_Custom_CNN.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\real_fake_ROC_Curve_Custom_CNN.pkl']

### ResNet50 Model:

In [41]:
fpr_tr, tpr_tr, _ = roc_curve(
                    resnet_train_results["labels"],
                    resnet_train_results["probs"][:, 1]      # probability of class 1 (real)
                )

fpr_va, tpr_va, _ = roc_curve(
                    resnet_val_results["labels"],
                    resnet_val_results["probs"][:, 1]
                )

fpr_te, tpr_te, _ = roc_curve(
                    resnet_test_results["labels"],
                    resnet_test_results["probs"][:, 1]
                )

fig = go.Figure()

fig.add_scatter(
                x=fpr_tr,
                y=tpr_tr,
                mode='lines',
                name=f'Train (AUC={resnet_train_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_va,
                y=tpr_va,
                mode='lines',
                name=f'Validation (AUC={resnet_val_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={resnet_test_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
                )

fig.update_layout(
                  height=500, width=800,
                  title='ROC Curve - ResNet50',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
                  )

fig.show()
joblib.dump(fig, CHARTS_DIR / "real_fake_ROC_Curve_ResNet50.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\real_fake_ROC_Curve_ResNet50.pkl']

### VGG16 Model:

In [42]:
fpr_tr, tpr_tr, _ = roc_curve(
                    vgg_train_results["labels"],
                    vgg_train_results["probs"][:, 1]      # probability of class 1 (real)
                )

fpr_va, tpr_va, _ = roc_curve(
                    vgg_val_results["labels"],
                    vgg_val_results["probs"][:, 1]
                )

fpr_te, tpr_te, _ = roc_curve(
                    vgg_test_results["labels"],
                    vgg_test_results["probs"][:, 1]
                )

fig = go.Figure()

fig.add_scatter(
                x=fpr_tr,
                y=tpr_tr,
                mode='lines',
                name=f'Train (AUC={vgg_train_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_va,
                y=tpr_va,
                mode='lines',
                name=f'Validation (AUC={vgg_val_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={vgg_test_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
                )

fig.update_layout(
                  height=500, width=800,
                  title='ROC Curve - VGG16',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
                  )

fig.show()
joblib.dump(fig, CHARTS_DIR / "real_fake_ROC_Curve_VGG16.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\real_fake_ROC_Curve_VGG16.pkl']

### EfficientNet_B0 Model:

In [43]:
fpr_tr, tpr_tr, _ = roc_curve(
                    effnet_train_results["labels"],
                    effnet_train_results["probs"][:, 1]      # probability of class 1 (real)
                )

fpr_va, tpr_va, _ = roc_curve(
                    effnet_val_results["labels"],
                    effnet_val_results["probs"][:, 1]
                )

fpr_te, tpr_te, _ = roc_curve(
                    effnet_test_results["labels"],
                    effnet_test_results["probs"][:, 1]
                )

fig = go.Figure()

fig.add_scatter(
                x=fpr_tr,
                y=tpr_tr,
                mode='lines',
                name=f'Train (AUC={effnet_train_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_va,
                y=tpr_va,
                mode='lines',
                name=f'Validation (AUC={effnet_val_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={effnet_test_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
                )

fig.update_layout(
                  height=500, width=800,
                  title='ROC Curve - EfficientNet-B0',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
                  )

fig.show()
joblib.dump(fig, CHARTS_DIR / "real_fake_ROC_Curve_EffNet-B0.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\real_fake_ROC_Curve_EffNet-B0.pkl']

### ViT_B16 Model:

In [44]:
fpr_tr, tpr_tr, _ = roc_curve(
                    vit_train_results["labels"],
                    vit_train_results["probs"][:, 1]      # probability of class 1 (real)
                )

fpr_va, tpr_va, _ = roc_curve(
                    vit_val_results["labels"],
                    vit_val_results["probs"][:, 1]
                )

fpr_te, tpr_te, _ = roc_curve(
                    vit_test_results["labels"],
                    vit_test_results["probs"][:, 1]
                )

fig = go.Figure()

fig.add_scatter(
                x=fpr_tr,
                y=tpr_tr,
                mode='lines',
                name=f'Train (AUC={vit_train_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_va,
                y=tpr_va,
                mode='lines',
                name=f'Validation (AUC={vit_val_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={vit_test_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
                )

fig.update_layout(
                  height=500, width=800,
                  title='ROC Curve - ViT-B/16',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
                  )

fig.show()
joblib.dump(fig, CHARTS_DIR / "real_fake_ROC_Curve_ViT_B16.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\real_fake\\real_fake_ROC_Curve_ViT_B16.pkl']

# Compare the models

The best model is selected using validation macro F1-score, not test performance. This keeps the test set reserved for final evaluation.

In [45]:
rows = []
for name, var in model_list:
    if var + "_test_results" in globals():
        val_r = globals()[var + "_val_results"]
        test_r = globals()[var + "_test_results"]
        rows.append({"Model": name,
                     "Val F1": round(val_r["f1"], 4),
                     "Test Accuracy": round(test_r["accuracy"], 4),
                     "Test F1": round(test_r["f1"], 4),
                     "Test AUC": round(test_r["auc"], 4)})

comparison = pd.DataFrame(rows).sort_values("Val F1", ascending=False).reset_index(drop=True)
comparison.to_csv(MODEL_DIR / "results.csv", index=False)
best_model_name = comparison.loc[0, "Model"]
print("Best model for", TASK, ":", best_model_name)
comparison

Best model for real_fake : CNN


,Model,Val F1,Test Accuracy,Test F1,Test AUC
0,CNN,1.0,1.0,1.0,1.0
1,ResNet50,1.0,1.0,1.0,1.0
2,VGG16,1.0,1.0,1.0,1.0
3,EfficientNet_B0,1.0,1.0,1.0,1.0
4,ViT_B16,1.0,1.0,1.0,1.0
